# Extract the DUACS (u,v) surface geostrophic velocities during the advection period

Computes an average of the velocity norm during the advection period (selecting weeks 25 to 29 + advection duration, 18 weeks).

Aim: extract the average geostrophic velocities at the position of the PF, which gives a proxy of the PF intensity (at the surface).

To read the DUACS data we use the loadCMEMSuv function coded in the LAMTA software (Louise Rousselet et al.)

##### Last version: Sept 2026 

In [294]:
import sys
sys.path.append('..')
import xarray as xr
from lamta.Diagnostics_mod import ParticleSet, Lagrangian, Eulerian
from lamta.Load_nc_mod import loadSWOTL3uv,loadCMEMSuv,loadsshuv,loadbathy
import numpy as np
from mpl_toolkits.basemap import Basemap #mapping toolbox
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib import cm
import cmocean as cm_oc
import netCDF4 as nc
from netCDF4 import Dataset
import datetime as dt
import os
import pandas as pd
import geopandas as gpd
import shapely
import tqdm
import datetime
from datetime import date, timedelta

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

import cmocean


### First step: find the dates that correspond to the advection period
We specifically select weeks 25 to 29 of each year (those shown in the article's main results).
We then add the advection duration (18 weeks) to the last week.
This leads us to extracting (u,v) roughly between the end of June and the end of November (for each year).

In [295]:
year=2023
# change here and rerun

In [296]:
### This bout of code uses datetime module and its functions

start_week_number=25 # week 24 + one week of vertical transport of the eggs
end_week_number=29 # idem with week 28

number_weeks_simu=18
numdays = number_weeks_simu*7 #18 weeks: duration of the advection


day_start = "{}-W{}".format(year,start_week_number) # use of datetime to find date corresponding to a week number
sdate = datetime.datetime.strptime(day_start + '-4', "%Y-W%W-%w").date()
print(type(sdate))
print(sdate)
day_end = "{}-W{}".format(year,end_week_number+number_weeks_simu) # last week + advection duration
edate=datetime.datetime.strptime(day_end + '-4', "%Y-W%W-%w").date()
print(type(edate))
print(edate)

<class 'datetime.date'>
2023-06-22
<class 'datetime.date'>
2023-11-23


In [297]:
all_days_pd=pd.date_range(sdate,edate-timedelta(days=1),freq='d')
print(len(all_days_pd))
all_days=[item.strftime('%Y%m%d') for item in all_days_pd] # make a list of dates in str format, between start and end dates
print(len(all_days))
print(all_days[0],all_days[-1])

154
154
20230622 20231122


### Load DUACS

Uses the CMEMSuv function coded in LAMTA.

In [298]:
# Loading altimetry (DUACS 1/8°) data
# Making use of the recently available 0.125° horizontal resolution DUACS data

rep='/home/analivaev/Documents/THESE/Data/DUACS_0125/{}/'.format(year) # directory where the data is stored

varn = {'longitude':'longitude','latitude':'latitude','u':'ugos','v':'vgos','ssh':'ssha'} # names of the variables we want to read


print(all_days)

field = loadCMEMSuv(all_days,rep,varn,unit='cm/s')


['20230622', '20230623', '20230624', '20230625', '20230626', '20230627', '20230628', '20230629', '20230630', '20230701', '20230702', '20230703', '20230704', '20230705', '20230706', '20230707', '20230708', '20230709', '20230710', '20230711', '20230712', '20230713', '20230714', '20230715', '20230716', '20230717', '20230718', '20230719', '20230720', '20230721', '20230722', '20230723', '20230724', '20230725', '20230726', '20230727', '20230728', '20230729', '20230730', '20230731', '20230801', '20230802', '20230803', '20230804', '20230805', '20230806', '20230807', '20230808', '20230809', '20230810', '20230811', '20230812', '20230813', '20230814', '20230815', '20230816', '20230817', '20230818', '20230819', '20230820', '20230821', '20230822', '20230823', '20230824', '20230825', '20230826', '20230827', '20230828', '20230829', '20230830', '20230831', '20230901', '20230902', '20230903', '20230904', '20230905', '20230906', '20230907', '20230908', '20230909', '20230910', '20230911', '20230912', '20

### Extract the data over the Kerguelen region, and average over time

In [299]:
lon_min,lon_max=60,80
ilon_min,ilon_max=np.searchsorted(field['lon'],[lon_min,lon_max])
lon_ker=np.arange(ilon_min,ilon_max+1)
#print(field['lon'][lon_ker])

lat_min,lat_max=-55,-45
ilat_min,ilat_max=np.searchsorted(field['lat'],[lat_min,lat_max])
lat_ker=np.arange(ilat_min,ilat_max+1)
#print(field['lat'][lat_ker])
X,Y=np.meshgrid(field['lon'][lon_ker],field['lat'][lat_ker])


[60.0625 60.1875 60.3125 60.4375 60.5625 60.6875 60.8125 60.9375 61.0625
 61.1875 61.3125 61.4375 61.5625 61.6875 61.8125 61.9375 62.0625 62.1875
 62.3125 62.4375 62.5625 62.6875 62.8125 62.9375 63.0625 63.1875 63.3125
 63.4375 63.5625 63.6875 63.8125 63.9375 64.0625 64.1875 64.3125 64.4375
 64.5625 64.6875 64.8125 64.9375 65.0625 65.1875 65.3125 65.4375 65.5625
 65.6875 65.8125 65.9375 66.0625 66.1875 66.3125 66.4375 66.5625 66.6875
 66.8125 66.9375 67.0625 67.1875 67.3125 67.4375 67.5625 67.6875 67.8125
 67.9375 68.0625 68.1875 68.3125 68.4375 68.5625 68.6875 68.8125 68.9375
 69.0625 69.1875 69.3125 69.4375 69.5625 69.6875 69.8125 69.9375 70.0625
 70.1875 70.3125 70.4375 70.5625 70.6875 70.8125 70.9375 71.0625 71.1875
 71.3125 71.4375 71.5625 71.6875 71.8125 71.9375 72.0625 72.1875 72.3125
 72.4375 72.5625 72.6875 72.8125 72.9375 73.0625 73.1875 73.3125 73.4375
 73.5625 73.6875 73.8125 73.9375 74.0625 74.1875 74.3125 74.4375 74.5625
 74.6875 74.8125 74.9375 75.0625 75.1875 75.3125 75

In [300]:
u_mean=np.nanmean(field['u'][:,ilon_min:ilon_max+1,ilat_min:ilat_max+1],axis=0).T
v_mean=np.nanmean(field['v'][:,ilon_min:ilon_max+1,ilat_min:ilat_max+1],axis=0).T
uv_norm=np.sqrt(u_mean**2+v_mean**2)
print(np.nanmean(uv_norm))

/tmp/ipykernel_5001/2846684555.py:1: RuntimeWarning: Mean of empty slice
  u_mean=np.nanmean(field['u'][:,ilon_min:ilon_max+1,ilat_min:ilat_max+1],axis=0).T
/tmp/ipykernel_5001/2846684555.py:2: RuntimeWarning: Mean of empty slice
  v_mean=np.nanmean(field['v'][:,ilon_min:ilon_max+1,ilat_min:ilat_max+1],axis=0).T


### Save to a netcdf file

In [302]:
def create_netcdf_2d(lon,lat,data,data_name,data_units,file_name,path_to_save):
    """Creates a NETCDF file from a 2 dimensional array (some time averaged data) with coordinates (lat, lon)"""
    ncout = Dataset(path_to_save+file_name,'w','NETCDF4') # using netCDF3 for output format 
    ncout.createDimension('lon',None)
    ncout.createDimension('lat',None)

    lonvar = ncout.createVariable('lon','float32',('lon'))
    lonvar[:] = lon[:]
    latvar = ncout.createVariable('lat','float32',('lat'))
    latvar[:] = lat[:]

    output_field= ncout.createVariable(data_name,'float32',('lat','lon'))
    output_field.setncattr('units',data_units)
    output_field[:,:] = data[:,:]
    ncout.close();

In [303]:
path_to_save='/home/analivaev/Documents/THESE/Data/Legines/averaged_velocity_fields/New_version_var_PF_position/'
file_name='{}_uv_norm_field.nc'.format(year)
create_netcdf_2d(field['lon'][lon_ker],field['lat'][lat_ker],uv_norm,'Geostrophic velocity field norm',data_units='cm/s',file_name=file_name,path_to_save=path_to_save)